# Предобработка данных о продажах компьютерных игр

- Автор: Наместникова Софья
- Дата: 17 сентября 2026 г.


## Описание проекта

Изучаем историю продаж компьютерных игр. Перед дальнейшим анализом надо проверить качество исходных данных, исправить ошибки, обработать пропуски и дубликаты, выделить актуальный период с 2000 по 2013 год и подготовить дополнительные категории оценок.

### Цель и задачи

**Цель:** подготовить данные о продажах игр к дальнейшему анализу рынка.

Для этого я:

1. загружу данные и изучу их структуру;
2. приведу названия столбцов и типы данных к корректному виду;
3. исследую и обработаю пропуски;
4. проверю явные и неявные дубликаты;
5. отберу игры, выпущенные с 2000 по 2013 год включительно;
6. категоризирую оценки пользователей и критиков;
7. определю семь платформ с наибольшим числом игр в актуальном периоде.

### Описание данных

Файл `/datasets/new_games.csv` содержит следующие поля:

- `Name` — название игры;
- `Platform` — игровая платформа;
- `Year of Release` — год выпуска;
- `Genre` — жанр;
- `NA sales`, `EU sales`, `JP sales`, `Other sales` — продажи в соответствующих регионах, млн копий;
- `Critic Score` — оценка критиков от 0 до 100;
- `User Score` — оценка пользователей от 0 до 10;
- `Rating` — возрастной рейтинг.

### Содержание проекта

1. Загрузка данных и знакомство с ними.
2. Проверка ошибок и предобработка.
3. Фильтрация актуального периода.
4. Категоризация оценок и определение топ-7 платформ.
5. Итоговый вывод.


---

## 1. Загрузка данных и знакомство с ними

Сначала импортирую pandas и загружу датасет. В учебной среде использую путь из задания, а ссылку оставляю запасным вариантом для локального запуска.

In [1]:
import pandas as pd


In [2]:
try:
    df = pd.read_csv('/datasets/new_games.csv')
except FileNotFoundError:
    df = pd.read_csv('https://code.s3.yandex.net/datasets/new_games.csv')

initial_rows = df.shape[0]
print(f'Размер исходного датасета: {initial_rows} строк и {df.shape[1]} столбцов')


Размер исходного датасета: 16956 строк и 11 столбцов


In [3]:
df.head()


,Name,Platform,Year of Release,Genre,NA sales,EU sales,JP sales,Other sales,Critic Score,User Score,Rating
0,Wii Sports,Wii,2006.0,Sports,41.36,28.96,3.77,8.45,76.0,8,E
1,Super Mario Bros.,NES,1985.0,Platform,29.08,3.58,6.81,0.77,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.0,Racing,15.68,12.76,3.79,3.29,82.0,8.3,E
3,Wii Sports Resort,Wii,2009.0,Sports,15.61,10.93,3.28,2.95,80.0,8,E
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,11.27,8.89,10.22,1.00,NaN,NaN,NaN


In [4]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             16954 non-null  str    
 1   Platform         16956 non-null  str    
 2   Year of Release  16681 non-null  float64
 3   Genre            16954 non-null  str    
 4   NA sales         16956 non-null  float64
 5   EU sales         16956 non-null  str    
 6   JP sales         16956 non-null  str    
 7   Other sales      16956 non-null  float64
 8   Critic Score     8242 non-null   float64
 9   User Score       10152 non-null  str    
 10  Rating           10085 non-null  str    
dtypes: float64(4), str(7)
memory usage: 1.4 MB


### Промежуточный вывод

В датасете 16 956 строк и 11 столбцов. Числовые представлены типами `float64` и `int64`, текстовые `object`. В `name`, `year_of_release`, `genre`, `critic_score`, `user_score` и `rating` есть пропуски. Столбцы `eu_sales`, `jp_sales` и `user_score`, несмотря на числовой смысл, имеют тип `object`: возмодно, в них присутствуют текстовые значения. Названия столбцов записаны с пробелами и заглавными буквами. На этапе предобработки проверю строковые значения, преобразую типы, изучу пропуски и дубликаты.


In [5]:
# создаем копию датасета до преобразования для возможности проверить сделанные изменения после предобработки, 
# не следует этот датасет использовать в проекте, он только для проверки.
tmp = df.copy() 
len(tmp)

16956

---

## 2. Проверка ошибок в данных и предобработка

### 2.1 Названия столбцов

Сначала проверю названия и приведу их к стилю `snake_case`.


In [6]:
print('Исходные названия столбцов:')
print(df.columns.tolist())

df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(r'\s+', '_', regex=True)
)

print('Названия после преобразования:')
print(df.columns.tolist())


Исходные названия столбцов:
['Name', 'Platform', 'Year of Release', 'Genre', 'NA sales', 'EU sales', 'JP sales', 'Other sales', 'Critic Score', 'User Score', 'Rating']
Названия после преобразования:
['name', 'platform', 'year_of_release', 'genre', 'na_sales', 'eu_sales', 'jp_sales', 'other_sales', 'critic_score', 'user_score', 'rating']


Названия привела к единому стилю: теперь они записаны в нижнем регистре, а пробелы заменены подчёркиваниями.


### 2.2 Типы данных

Проверю, какие текстовые значения мешают преобразованию числовых столбцов.


In [7]:
numeric_as_text = ['eu_sales', 'jp_sales', 'user_score']

for column in numeric_as_text:
    converted = pd.to_numeric(df[column], errors='coerce')
    invalid_mask = converted.isna() & df[column].notna()
    print(f'{column}:')
    print(df.loc[invalid_mask, column].value_counts())
    print()


eu_sales:
eu_sales
unknown    6
Name: count, dtype: int64

jp_sales:
jp_sales
unknown    4
Name: count, dtype: int64

user_score:
user_score
tbd    2464
Name: count, dtype: int64



В `eu_sales` найдено 6 значений `unknown`, в `jp_sales` 4. Они означают неизвестное количество продаж. В `user_score` найдено 2464 значения `tbd`: оценка ещё не определена. Эти значения нельзя интерпретировать как ноль, поэтому преобразую их в `NaN` с помощью `errors='coerce'`.


In [8]:
df['year_of_release'] = pd.to_numeric(df['year_of_release'], errors='coerce')
df['eu_sales'] = pd.to_numeric(df['eu_sales'], errors='coerce')
df['jp_sales'] = pd.to_numeric(df['jp_sales'], errors='coerce')
df['user_score'] = pd.to_numeric(df['user_score'], errors='coerce')

df.dtypes


name                   str
platform               str
year_of_release    float64
genre                  str
na_sales           float64
eu_sales           float64
jp_sales           float64
other_sales        float64
critic_score       float64
user_score         float64
rating                 str
dtype: object

После преобразования `eu_sales`, `jp_sales` и `user_score` получили числовой тип `float64`. Год выпуска пока остаётся `float64`, поскольку в нём есть пропуски; после их обработки преобразую его в `int64`.


### 2.3 Пропущенные значения

Посчитаю пропуски в абсолютных и относительных значениях.


In [9]:
missing_stats = pd.DataFrame({
    'missing_count': df.isna().sum(),
    'missing_share_pct': (df.isna().mean() * 100).round(2)
})
missing_stats[missing_stats['missing_count'] > 0]


,missing_count,missing_share_pct
name,2,0.01
year_of_release,275,1.62
genre,2,0.01
eu_sales,6,0.04
jp_sales,4,0.02
critic_score,8714,51.39
user_score,9268,54.66
rating,6871,40.52


In [10]:
def show_missing_stats(tmp0):
    """
    Функция для отображения статистики пропущенных значений в DataFrame.
    """
    missing_stats = pd.DataFrame({
        'Кол-во пропусков': tmp0.isnull().sum(),
        'Доля пропусков': tmp0.isnull().mean()
    })
    missing_stats = missing_stats[missing_stats['Кол-во пропусков'] > 0]
    
    if missing_stats.empty:
        return "Пропусков в данных нет"
    
    # Форматируем при выводе через Styler
    return (missing_stats.style.format({'Доля пропусков': '{:.4f}'}).background_gradient(cmap='coolwarm'))
show_missing_stats(df)

,Кол-во пропусков,Доля пропусков
name,2,0.0001
year_of_release,275,0.0162
genre,2,0.0001
eu_sales,6,0.0004
jp_sales,4,0.0002
critic_score,8714,0.5139
user_score,9268,0.5466
rating,6871,0.4052


Больше всего пропусков в `critic_score` (около 51%), `user_score` (около 55%) и `rating` (около 41%). В `year_of_release` пропущено примерно 1,6% значений. После преобразования текстовых маркеров появились 6 пропусков в `eu_sales` и 4 в `jp_sales`. В `name` и `genre` отсутствует по 2 значения.

Возможные причины: оценки могли отсутствовать у игр без достаточного количества отзывов; `tbd` обозначает ещё не сформированную пользовательскую оценку; рейтинг ESRB присваивается для рынка США и Канады, поэтому его отсутствие может быть содержательной характеристикой, а не ошибкой; `unknown` в продажах означает, что значение не определено.


#### Обработка пропусков в названии, жанре и годе выпуска

Строки без названия и жанра невозможно идентифицировать. Год выпуска нужен для фильтрации актуального периода и заполнения продаж по платформе и году, поэтому строки без него также удалю. Сначала посмотрю на эти наблюдения.


In [11]:
df.loc[
    df['name'].isna() | df['genre'].isna() | df['year_of_release'].isna(),
    ['name', 'platform', 'year_of_release', 'genre', 'critic_score', 'user_score', 'rating']
].head(10)


,name,platform,year_of_release,genre,critic_score,user_score,rating
183,Madden NFL 2004,PS2,NaN,Sports,94.0,8.5,E
379,FIFA Soccer 2004,PS2,NaN,Sports,84.0,6.4,E
458,LEGO Batman: The Videogame,Wii,NaN,Action,74.0,7.9,E10+
477,wwe Smackdown vs. Raw 2006,PS2,NaN,Fighting,NaN,NaN,NaN
611,Space Invaders,2600,NaN,Shooter,NaN,NaN,NaN
629,Rock Band,X360,NaN,Misc,92.0,8.2,T
659,Frogger's Adventures: Temple of the Frog,GBA,NaN,Adventure,73.0,NaN,E
661,NaN,GEN,1993.0,NaN,NaN,NaN,NaN
680,LEGO Indiana Jones: The Original Adventures,Wii,NaN,Action,78.0,6.6,E10+
722,Call of Duty 3,Wii,NaN,Shooter,69.0,6.7,T


In [12]:
df = df.dropna(subset=['name', 'genre', 'year_of_release']).copy()
df['year_of_release'] = df['year_of_release'].astype('int64')

print(f'После удаления строк без названия, жанра или года осталось {len(df)} строк')


После удаления строк без названия, жанра или года осталось 16679 строк


#### Обработка пропусков в региональных продажах

Пропуски в `eu_sales` и `jp_sales` заполню средним значением внутри группы «платформа — год выпуска». Такая группировка учитывает различия между поколениями платформ и периодами их активности.


In [13]:
for column in ['eu_sales', 'jp_sales']:
    missing_before = df[column].isna().sum()
    group_mean = df.groupby(['platform', 'year_of_release'])[column].transform('mean')
    df[column] = df[column].fillna(group_mean)
    print(f'{column}: пропусков до — {missing_before}, после — {df[column].isna().sum()}')


eu_sales: пропусков до — 6, после — 0
jp_sales: пропусков до — 4, после — 0


Все пропуски в продажах удалось заполнить средними значениями по соответствующей платформе и году.

#### Обработка пропусков в оценках и возрастном рейтинге

Пропуски в `critic_score` и `user_score` оставлю как `NaN`: замена средним исказила бы реальные оценки и категоризацию. Пропуски `rating` заменю на `UNKNOWN`, поскольку отсутствие ESRB-рейтинга может означать, что игра выпускалась не для североамериканского рынка.


In [14]:
df['rating'] = df['rating'].fillna('UNKNOWN')

show_missing_stats(df)


,Кол-во пропусков,Доля пропусков
critic_score,8594,0.5153
user_score,9121,0.5469


In [15]:
# Посмотрим какие пропуски остались
show_missing_stats(df)

,Кол-во пропусков,Доля пропусков
critic_score,8594,0.5153
user_score,9121,0.5469


После обработки пропуски остались только в оценках критиков и пользователей. Они отражают отсутствие оценки и не заменялись искусственными значениями.


### 2.4 Явные и неявные дубликаты

Проверю уникальные значения жанров, платформ, рейтингов и лет выпуска. Текстовые значения нормализую: жанры приведу к нижнему регистру, платформы и рейтинги — к верхнему.


In [16]:
for column in ['platform', 'genre', 'rating']:
    print(f'{column}: {sorted(df[column].dropna().unique().tolist())}')

print(f'Годы выпуска: от {df["year_of_release"].min()} до {df["year_of_release"].max()}')


platform: ['2600', '3DO', '3DS', 'DC', 'DS', 'GB', 'GBA', 'GC', 'GEN', 'GG', 'N64', 'NES', 'NG', 'PC', 'PCFX', 'PS', 'PS2', 'PS3', 'PS4', 'PSP', 'PSV', 'SAT', 'SCD', 'SNES', 'TG16', 'WS', 'Wii', 'WiiU', 'X360', 'XB', 'XOne']
genre: ['ACTION', 'ADVENTURE', 'Action', 'Adventure', 'FIGHTING', 'Fighting', 'MISC', 'Misc', 'PLATFORM', 'PUZZLE', 'Platform', 'Puzzle', 'RACING', 'ROLE-PLAYING', 'Racing', 'Role-Playing', 'SHOOTER', 'SIMULATION', 'SPORTS', 'STRATEGY', 'Shooter', 'Simulation', 'Sports', 'Strategy']
rating: ['AO', 'E', 'E10+', 'EC', 'K-A', 'M', 'RP', 'T', 'UNKNOWN']
Годы выпуска: от 1980 до 2016


В `genre` обнаружены варианты капсом. Названия платформ не содержат дублей, но для единообразия я также сделаю капсом. Рейтинг `UNKNOWN` сохраню отдельной категорией.


In [17]:
df['name'] = df['name'].str.strip()
df['genre'] = df['genre'].str.lower().str.strip()
df['platform'] = df['platform'].str.upper().str.strip()
df['rating'] = df['rating'].str.upper().str.strip()

for column in ['platform', 'genre', 'rating']:
    print(f'{column}: {sorted(df[column].unique().tolist())}')


platform: ['2600', '3DO', '3DS', 'DC', 'DS', 'GB', 'GBA', 'GC', 'GEN', 'GG', 'N64', 'NES', 'NG', 'PC', 'PCFX', 'PS', 'PS2', 'PS3', 'PS4', 'PSP', 'PSV', 'SAT', 'SCD', 'SNES', 'TG16', 'WII', 'WIIU', 'WS', 'X360', 'XB', 'XONE']
genre: ['action', 'adventure', 'fighting', 'misc', 'platform', 'puzzle', 'racing', 'role-playing', 'shooter', 'simulation', 'sports', 'strategy']
rating: ['AO', 'E', 'E10+', 'EC', 'K-A', 'M', 'RP', 'T', 'UNKNOWN']


In [18]:
duplicates_count = df.duplicated().sum()
print(f'Явных дубликатов после нормализации: {duplicates_count}')

df = df.drop_duplicates().reset_index(drop=True)


Явных дубликатов после нормализации: 235


После нормализации найдено 235 полных дубликатов. Удалила поскольку полностью повторяют другие строки и не несут дополнительной информации.


In [19]:
removed_rows = initial_rows - len(df)
print(f'Исходное число строк: {initial_rows}')
print(f'После предобработки: {len(df)}')
print(f'Удалено строк: {removed_rows} ({removed_rows / initial_rows:.2%})')


Исходное число строк: 16956
После предобработки: 16444
Удалено строк: 512 (3.02%)


### Общий промежуточный вывод по предобработке

Я привела названия столбцов к `snake_case`, изучила специальные значения `unknown` и `tbd` и преобразовала числовые столбцы. Удалила строки без названия, жанра или года выпуска, заполнила 10 пропусков региональных продаж средними по платформе и году, оставила отсутствующие оценки как `NaN`, а неизвестный ESRB-рейтинг обозначила `UNKNOWN`. Жанры и другие категориальные поля нормализованы, 235 полных дубликатов удалены. Всего удалено 512 строк.


---

## 3. Фильтрация данных

Отберу игры, выпущенные с 2000 по 2013 год включительно, и сохраню результат в отдельном датафрейме `df_actual`.


In [20]:
df_actual = df[df['year_of_release'].between(2000, 2013, inclusive='both')].copy()

print(f'Размер актуального среза: {df_actual.shape[0]} строк и {df_actual.shape[1]} столбцов')
print(f'Период: {df_actual["year_of_release"].min()}–{df_actual["year_of_release"].max()} годы')


Размер актуального среза: 12781 строк и 11 столбцов
Период: 2000–2013 годы


В актуальный срез вошла 12 781 игра, выпущенная в период с 2000 по 2013 год включительно.


---

## 4. Категоризация данных

Добавлю категории оценок пользователей и критиков. Игры без оценки не буду ошибочно относить к низкой категории: для них сохраню `NaN`.


In [21]:
def categorize_user_score(score):
    if pd.isna(score):
        return pd.NA
    if 8 <= score <= 10:
        return 'высокая оценка'
    if 3 <= score < 8:
        return 'средняя оценка'
    if 0 <= score < 3:
        return 'низкая оценка'
    return pd.NA


def categorize_critic_score(score):
    if pd.isna(score):
        return pd.NA
    if 80 <= score <= 100:
        return 'высокая оценка'
    if 30 <= score < 80:
        return 'средняя оценка'
    if 0 <= score < 30:
        return 'низкая оценка'
    return pd.NA


df_actual['user_score_category'] = df_actual['user_score'].map(categorize_user_score)
df_actual['critic_score_category'] = df_actual['critic_score'].map(categorize_critic_score)


Проверю результат категоризации, включая количество игр без оценок.


In [22]:
print('Категории пользовательских оценок:')
print(df_actual['user_score_category'].value_counts(dropna=False))

print('\nКатегории оценок критиков:')
print(df_actual['critic_score_category'].value_counts(dropna=False))


Категории пользовательских оценок:
user_score_category
NaN               6298
средняя оценка    4081
высокая оценка    2286
низкая оценка      116
Name: count, dtype: int64

Категории оценок критиков:
critic_score_category
NaN               5612
средняя оценка    5422
высокая оценка    1692
низкая оценка       55
Name: count, dtype: int64


Среди известных пользовательских оценок преобладает средняя категория — 4081 игра; высокую оценку получили 2286 игр, низкую — 116. Пользовательской оценки нет у 6298 игр. Среди оценок критиков также преобладает средняя категория — 5422 игры; высокую оценку получили 1692 игры, низкую — 55, а оценка отсутствует у 5612 игр.


### Топ-7 платформ актуального периода

Посчитаю количество игр для каждой платформы за 2000–2013 годы.


In [23]:
top_platforms = df_actual['platform'].value_counts().head(7)
top_platforms


platform
PS2     2127
DS      2120
WII     1275
PSP     1180
X360    1121
PS3     1087
GBA      811
Name: count, dtype: int64

В топ-7 вошли PS2, DS, Wii, PSP, X360, PS3 и GBA. Лидируют PS2 и DS: для них в актуальном периоде представлено более двух тысяч игр.


---

## 5. Итоговый вывод

Я загрузила датасет из 16 956 строк и 11 столбцов и провела весь цикл предобработки. Названия столбцов привела к `snake_case`. В числовых полях обнаружила специальные значения: `unknown` в европейских и японских продажах и `tbd` в пользовательских оценках; после анализа преобразовала их в пропуски и исправила типы данных.

Строки без названия, жанра или года выпуска удалила. Пропуски в региональных продажах заполнила средними значениями по платформе и году выпуска. Отсутствующие оценки критиков и пользователей оставила как `NaN`, чтобы не искажать данные, а отсутствие ESRB-рейтинга обозначила `UNKNOWN`. После нормализации текстовых значений удалила 235 полных дубликатов. Всего при предобработке удалено 512 строк, или около 3,02% исходных данных.

Для анализа подготовила отдельный срез `df_actual` из 12 781 игры за 2000–2013 годы. В него добавлены два поля: `user_score_category` и `critic_score_category` с высокой, средней и низкой оценкой; для игр без оценки сохранены пропуски. Чаще всего известные оценки относятся к средней категории. Топ-7 платформ по количеству игр в актуальном периоде составили PS2, DS, Wii, PSP, X360, PS3 и GBA.
